# Research Results and Statistical Analysis

## Comparative Evaluation of Machine Learning Models
## for Flow-Based DDoS Attack Detection

This notebook presents the reproducible experimental
analysis of machine learning benchmark measurements.

The analysis includes:

- Raw data ingestion.
- Warm-up filtering.
- Data quality validation.
- Error-rate monitoring.
- Statistical aggregation.
- Median, p95 and p99 calculation.
- Bootstrap 95% confidence intervals.
- Scientific visualization.
- Exporting processed results.

Random seed: 42.

In [1]:
import sys
import json
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

current = Path.cwd().resolve()

ROOT = None

for candidate in [current, *current.parents]:
    if (
        (candidate / "data").exists()
        and (candidate / "src").exists()
    ):
        ROOT = candidate
        break

if ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root."
    )

RAW_PATH = ROOT / "data/raw/test_run.jsonl"

PROCESSED_PATH = (
    ROOT / "data/processed/metrics_summary.csv"
)

FIGURES_PATH = ROOT / "reports/figures"

FIGURES_PATH.mkdir(
    parents=True,
    exist_ok=True
)

sns.set_theme(
    style="whitegrid",
    context="paper",
    palette="colorblind"
)

print("Project root:", ROOT)
print("Random seed:", SEED)
print("Raw data:", RAW_PATH)

Project root: C:\Users\aerul\Desktop\task 2\research_methods-
Random seed: 42
Raw data: C:\Users\aerul\Desktop\task 2\research_methods-\data\raw\test_run.jsonl


## 1. Data Ingestion

The raw benchmark data is stored in JSON Lines format.

Each line represents one benchmark measurement.

The dataset contains warm-up and evaluation records.

Warm-up records are excluded from the final analysis
to reduce the influence of initial execution overhead.

In [2]:
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Raw data not found: {RAW_PATH}"
    )

records = []

with RAW_PATH.open("r", encoding="utf-8") as file:

    for line_number, line in enumerate(file, start=1):

        if not line.strip():
            continue

        try:
            records.append(json.loads(line))

        except json.JSONDecodeError as error:

            raise ValueError(
                f"Invalid JSON at line {line_number}"
            ) from error

df = pd.DataFrame(records)

print("Total raw records:", len(df))

print("\nAvailable columns:")
print(df.columns.tolist())

display(df.head())

Total raw records: 12000

Available columns:
['run_id', 'fold', 'model', 'phase', 'is_warmup', 'load_rps', 'batch_size', 'latency_ms', 'throughput_rps', 'accuracy', 'precision', 'recall', 'f1_score', 'fpr', 'error', 'status_code']


,run_id,fold,model,phase,is_warmup,load_rps,batch_size,latency_ms,throughput_rps,accuracy,precision,recall,f1_score,fpr,error,status_code
0,fold_1_Logistic Regression_1_0,1,Logistic Regression,warmup,True,1,1,2.6131,1913.436156,1.0,1.0,1.0,1.0,0.0,False,200
1,fold_1_Logistic Regression_1_1,1,Logistic Regression,warmup,True,1,1,2.1714,2302.661874,1.0,1.0,1.0,1.0,0.0,False,200
2,fold_1_Logistic Regression_1_2,1,Logistic Regression,warmup,True,1,1,2.0825,2400.960490,1.0,1.0,1.0,1.0,0.0,False,200
3,fold_1_Logistic Regression_1_3,1,Logistic Regression,warmup,True,1,1,2.0356,2456.278367,1.0,1.0,1.0,1.0,0.0,False,200
4,fold_1_Logistic Regression_1_4,1,Logistic Regression,warmup,True,1,1,4.0470,1235.483120,1.0,1.0,1.0,1.0,0.0,False,200


## 2. Warm-up Filtering

The initial execution measurements are classified
as warm-up runs.

These records are excluded from statistical analysis.

Only records with phase = evaluation and
is_warmup = False are retained.

In [3]:
def to_bool(series):

    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )


df["is_warmup"] = to_bool(df["is_warmup"])

df["error"] = to_bool(df["error"])

df["phase"] = (
    df["phase"]
    .astype(str)
    .str.lower()
)

warmup_count = (
    (df["phase"] == "warmup") |
    df["is_warmup"]
).sum()

df_eval = df[
    (df["phase"] == "evaluation") &
    (~df["is_warmup"])
].copy()

print("Total raw records:", len(df))

print("Warm-up records removed:", int(warmup_count))

print("Evaluation records:", len(df_eval))

assert len(df_eval) > 0

assert not df_eval["is_warmup"].any()

display(df_eval.head())

Total raw records: 12000
Warm-up records removed: 2400
Evaluation records: 9600


,run_id,fold,model,phase,is_warmup,load_rps,batch_size,latency_ms,throughput_rps,accuracy,precision,recall,f1_score,fpr,error,status_code
5,fold_1_Logistic Regression_1_5,1,Logistic Regression,evaluation,False,1,1,2.6184,1909.563154,1.0,1.0,1.0,1.0,0.0,False,200
6,fold_1_Logistic Regression_1_6,1,Logistic Regression,evaluation,False,1,1,2.2016,2271.075581,1.0,1.0,1.0,1.0,0.0,False,200
7,fold_1_Logistic Regression_1_7,1,Logistic Regression,evaluation,False,1,1,2.1835,2289.901556,1.0,1.0,1.0,1.0,0.0,False,200
8,fold_1_Logistic Regression_1_8,1,Logistic Regression,evaluation,False,1,1,2.0835,2399.808112,1.0,1.0,1.0,1.0,0.0,False,200
9,fold_1_Logistic Regression_1_9,1,Logistic Regression,evaluation,False,1,1,2.0625,2424.242164,1.0,1.0,1.0,1.0,0.0,False,200


## 3. Data Quality and Error Analysis

Failed inference measurements are monitored
as a guardrail metric.

The error rate is calculated as:

Error Rate = Failed Runs / Total Evaluation Runs

Only successful inference measurements are used
for latency and throughput calculations.

In [4]:
total_runs = len(df_eval)

failed_runs = int(
    df_eval["error"].sum()
)

error_rate = (
    failed_runs / total_runs
    if total_runs > 0
    else np.nan
)

print("Total evaluation runs:", total_runs)

print("Failed evaluation runs:", failed_runs)

print(f"Overall error rate: {error_rate:.4%}")

error_summary = (
    df_eval.groupby("model")
    .agg(
        total_runs=("error", "count"),
        errors=("error", "sum")
    )
)

error_summary["error_rate"] = (
    error_summary["errors"] /
    error_summary["total_runs"]
)

display(error_summary)

Total evaluation runs: 9600
Failed evaluation runs: 0
Overall error rate: 0.0000%


,total_runs,errors,error_rate
model,,,
Decision Tree,2400,0,0.0
Logistic Regression,2400,0,0.0
Random Forest,2400,0,0.0
XGBoost,2400,0,0.0


## 4. Statistical Aggregation

The following statistics are calculated:

- Median latency.
- 95th percentile latency.
- 99th percentile latency.
- Bootstrap 95% confidence interval.
- Median throughput.
- Classification metrics.
- Error rate.

The bootstrap procedure uses 5000 resamples
and a fixed random seed of 42.

In [5]:
def bootstrap_ci(
    values,
    iterations=5000,
    statistic=np.median
):

    values = np.asarray(
        values,
        dtype=float
    )

    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan

    if len(values) == 1:
        return float(values[0]), float(values[0])

    rng = np.random.default_rng(SEED)

    samples = rng.choice(
        values,
        size=(iterations, len(values)),
        replace=True
    )

    estimates = statistic(
        samples,
        axis=1
    )

    return (
        float(np.percentile(estimates, 2.5)),
        float(np.percentile(estimates, 97.5))
    )


successful = df_eval[
    ~df_eval["error"]
].copy()

successful["latency_ms"] = pd.to_numeric(
    successful["latency_ms"],
    errors="coerce"
)

successful["throughput_rps"] = pd.to_numeric(
    successful["throughput_rps"],
    errors="coerce"
)

successful = successful.replace(
    [np.inf, -np.inf],
    np.nan
)

successful = successful.dropna(
    subset=["latency_ms", "throughput_rps"]
)

latency_results = []

for (model, batch), group in successful.groupby(
    ["model", "batch_size"]
):

    values = group["latency_ms"].to_numpy()

    low, high = bootstrap_ci(values)

    latency_results.append({

        "model": model,

        "batch_size": batch,

        "count": len(values),

        "median_latency_ms": np.median(values),

        "p95_latency_ms": np.percentile(values, 95),

        "p99_latency_ms": np.percentile(values, 99),

        "ci95_low": low,

        "ci95_high": high

    })

latency_summary = pd.DataFrame(
    latency_results
)

display(latency_summary)

,model,batch_size,count,median_latency_ms,p95_latency_ms,p99_latency_ms,ci95_low,ci95_high
0,Decision Tree,1,400,1.80960,2.985440,4.026330,1.771200,1.853850
1,Decision Tree,2,400,1.74090,2.818025,2.969640,1.723300,1.770800
2,Decision Tree,4,400,1.76135,2.821460,3.121834,1.740000,1.793550
3,Decision Tree,8,400,1.72225,2.618025,2.834342,1.708295,1.732300
4,Decision Tree,16,400,1.77130,2.729365,2.931931,1.747700,1.798700
5,Decision Tree,32,400,1.80595,2.859465,3.087105,1.774700,1.879800
6,Logistic Regression,1,400,2.20245,3.380305,4.165645,2.164000,2.254700
7,Logistic Regression,2,400,2.11675,3.501195,3.839153,2.100400,2.137550
8,Logistic Regression,4,400,2.11270,3.362795,3.714156,2.082800,2.138250
9,Logistic Regression,8,400,2.09855,3.228580,3.527651,2.082900,2.122601


In [6]:
classification_results = []

for model, group in df_eval.groupby("model"):

    folds = group.drop_duplicates(
        subset=["fold"]
    )

    for metric in [
        "accuracy",
        "precision",
        "recall",
        "f1_score",
        "fpr"
    ]:

        values = (
            pd.to_numeric(
                folds[metric],
                errors="coerce"
            )
            .dropna()
            .to_numpy()
        )

        if len(values) == 0:
            continue

        low, high = bootstrap_ci(
            values,
            statistic=np.mean
        )

        classification_results.append({

            "model": model,

            "metric": metric,

            "mean": np.mean(values),

            "ci95_low": low,

            "ci95_high": high

        })

classification_summary = pd.DataFrame(
    classification_results
)

display(classification_summary)

,model,metric,mean,ci95_low,ci95_high
0,Decision Tree,accuracy,1.000000,1.000000,1.00
1,Decision Tree,precision,1.000000,1.000000,1.00
2,Decision Tree,recall,1.000000,1.000000,1.00
3,Decision Tree,f1_score,1.000000,1.000000,1.00
4,Decision Tree,fpr,0.000000,0.000000,0.00
5,Logistic Regression,accuracy,1.000000,1.000000,1.00
6,Logistic Regression,precision,1.000000,1.000000,1.00
7,Logistic Regression,recall,1.000000,1.000000,1.00
8,Logistic Regression,f1_score,1.000000,1.000000,1.00
9,Logistic Regression,fpr,0.000000,0.000000,0.00


## 5. Visualization

The first figure presents median inference latency
under different batch multipliers.

The shaded region represents the bootstrap
95% confidence interval of the median.

The second figure compares latency distributions
at a fixed batch multiplier of 1.

Both figures are exported in PDF and PNG formats.

In [7]:
subprocess.run(
    [
        sys.executable,
        str(ROOT / "src/process_data.py")
    ],
    check=True,
    cwd=ROOT
)

final_summary = pd.read_csv(
    PROCESSED_PATH
)

print("Processed statistical summary:")

display(final_summary)

Processed statistical summary:


,model,batch_size,n_evaluations,n_successful,median_latency_ms,p95_latency_ms,p99_latency_ms,latency_ci95_low,latency_ci95_high,median_throughput_samples_s,throughput_ci95_low,throughput_ci95_high,mean_f1_score,f1_ci95_low,f1_ci95_high,mean_fpr,fpr_ci95_low,fpr_ci95_high,error_count,error_rate
0,Decision Tree,1,400,400,1.80960,2.985440,4.026330,1.771200,1.853850,2763.042353,2697.089899,2822.944924,1.000000,1.000000,1.0,0.0,0.0,0.0,0,0.0
1,Decision Tree,2,400,400,1.74090,2.818025,2.969640,1.723300,1.770800,5744.158486,5647.165188,5802.820359,1.000000,1.000000,1.0,0.0,0.0,0.0,0,0.0
2,Decision Tree,4,400,400,1.76135,2.821460,3.121834,1.740000,1.793550,11354.930874,11151.072343,11494.251482,1.000000,1.000000,1.0,0.0,0.0,0.0,0,0.0
3,Decision Tree,8,400,400,1.72225,2.618025,2.834342,1.708295,1.732300,23225.433199,23090.688205,23415.161289,1.000000,1.000000,1.0,0.0,0.0,0.0,0,0.0
4,Decision Tree,16,400,400,1.77130,2.729365,2.931931,1.747700,1.798700,45164.581001,44476.564540,45774.450852,1.000000,1.000000,1.0,0.0,0.0,0.0,0,0.0
5,Decision Tree,32,400,400,1.80595,2.859465,3.087105,1.774700,1.879800,88596.032008,85115.443326,90156.076334,1.000000,1.000000,1.0,0.0,0.0,0.0,0,0.0
6,Logistic Regression,1,400,400,2.20245,3.380305,4.165645,2.164000,2.254700,2270.199368,2217.592291,2310.541154,1.000000,1.000000,1.0,0.0,0.0,0.0,0,0.0
7,Logistic Regression,2,400,400,2.11675,3.501195,3.839153,2.100400,2.137550,4724.223232,4678.253493,4760.998068,1.000000,1.000000,1.0,0.0,0.0,0.0,0,0.0
8,Logistic Regression,4,400,400,2.11270,3.362795,3.714156,2.082800,2.138250,9466.559366,9353.451141,9602.458105,1.000000,1.000000,1.0,0.0,0.0,0.0,0,0.0
9,Logistic Regression,8,400,400,2.09855,3.228580,3.527651,2.082900,2.122601,19060.781086,18844.801112,19203.997009,1.000000,1.000000,1.0,0.0,0.0,0.0,0,0.0


In [8]:
subprocess.run(
    [
        sys.executable,
        str(ROOT / "src/visualize.py")
    ],
    check=True,
    cwd=ROOT
)

print("Generated figures:")

for figure in sorted(FIGURES_PATH.iterdir()):

    if figure.suffix in [".pdf", ".png"]:

        print(
            figure.name,
            f"({figure.stat().st_size:,} bytes)"
        )

Generated figures:
fig1_latency_vs_load.pdf (21,917 bytes)
fig1_latency_vs_load.png (161,730 bytes)
fig2_distribution_boxplot.pdf (22,258 bytes)
fig2_distribution_boxplot.png (137,239 bytes)


## 6. Results and Discussion

The experiment compares four machine learning
classifiers in terms of computational performance
and classification quality.

The statistical analysis includes median latency,
p95, p99, throughput and error rates.

Warm-up measurements are excluded from the analysis.

The results should be interpreted with caution.

The original classification dataset contains only
ten synthetic observations. Therefore, the experiment
does not establish reliable real-world DDoS detection
performance.

Larger and more representative network traffic
datasets are required for further validation.

## 7. Reproducibility

The analysis uses:

- Python.
- NumPy.
- Pandas.
- Matplotlib.
- Seaborn.
- Bootstrap resampling.
- Random seed 42.

The raw data, processing scripts and generated figures
are stored in the project repository.